<a href="https://colab.research.google.com/github/jygheo/Contrastive-Decoding/blob/main/music_generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
from google.colab import runtime
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import gc
import torch
import torchaudio.functional as F_audio
import torch.nn.functional as F
from datasets import load_dataset, disable_progress_bar
from datasets.utils.logging import set_verbosity_error
from transformers import MusicgenForConditionalGeneration, AutoProcessor, LogitsProcessorList, LogitsProcessor, set_seed
from transformers.utils.logging import set_verbosity_error as set_tf_verbosity_error
from transformers.modeling_outputs import BaseModelOutput
from transformers import LogitsProcessor
import random
import os
import json
import time
import contextlib
import functools

set_seed(42)
set_verbosity_error()
set_tf_verbosity_error()
disable_progress_bar()

In [ ]:
def to_model_device_dtype(x, model):
    return x.to(
        device=next(model.parameters()).device,
        dtype=next(model.parameters()).dtype
    )

In [ ]:
def extract_prompt_and_reference(raw_audio, processor, expert_model, prompt_sec=5, target_sec=10):
    """
    Slices raw audio, resamples, and encodes into EnCodec tokens
    """
    sr = raw_audio['sampling_rate']
    wav = torch.tensor(raw_audio['array'], dtype=torch.float32)

    # convert stereo -> mono
    if wav.ndim > 1 and wav.shape[0] == 2:
        wav = wav.mean(dim=0)
    elif wav.ndim == 1:
        wav = wav.unsqueeze(0) # (1, length)

    # resample to 32kHz
    target_sr = processor.feature_extractor.sampling_rate
    if sr != target_sr:
        wav = F_audio.resample(wav, sr, target_sr)

    total_required_frames = (prompt_sec + target_sec) * target_sr
    if wav.shape[1] < total_required_frames:
        return None, None

    prompt_wav = to_model_device_dtype(wav[:, :prompt_sec * target_sr].unsqueeze(0),expert_model)
    ref_wav = to_model_device_dtype(wav[:, prompt_sec * target_sr : total_required_frames].unsqueeze(0), expert_model)

    with torch.no_grad():
        prompt_outputs = expert_model.audio_encoder.encode(prompt_wav)
        ref_outputs = expert_model.audio_encoder.encode(ref_wav)
        prompt_tokens = prompt_outputs.audio_codes[0, 0]
        ref_tokens = ref_outputs.audio_codes[0, 0]
    del wav, prompt_wav, ref_wav, prompt_outputs, ref_outputs

    return prompt_tokens.cpu().tolist(), ref_tokens.cpu().tolist()

In [ ]:
def get_ds(dataset_path, processor, expert_model, dataset_name=None, split="train", target_count=1000):
    print(f"Downloading {dataset_path} ({split})")

    ds = load_dataset(dataset_path, dataset_name, split=split, streaming=True)
    ds = ds.shuffle(seed=42, buffer_size=1000)
    valid_samples = []

    anti_genre_map = {
        'blues': 'math rock',
        'classical': 'dubstep',
        'country': 'ambient lofi',
        'disco': 'funeral march',
        'hiphop': 'classical symphony',
        'jazz': 'heavy metal',
        'metal': 'smooth jazz',
        'pop': 'dissonant avant-garde',
        'reggae': 'classical piano',
        'rock': 'lullaby'
    }

    labels = ['blues', 'classical', 'country', 'disco', 'hiphop', 'jazz', 'metal', 'pop', 'reggae', 'rock']

    for row in ds:
        raw_audio = row.get('audio')
        label_num = row.get('genre')
        if label_num is None:
            continue

        label = labels[label_num]
        anti_label = anti_genre_map[label]

        text_prompt = f'Continue this {label} piece'
        anti_text_prompt = f'Continue this {anti_label} piece'

        prompt, reference = extract_prompt_and_reference(raw_audio, processor, expert_model)

        if prompt and reference:
            valid_samples.append({
                "prompt": prompt,
                "text_prompt": text_prompt,
                "anti_text_prompt": anti_text_prompt, # Store the anti-prompt here
                "reference": reference
            })

        if len(valid_samples) >= target_count:
            print(f"Target of {target_count} reached")
            break

    del ds
    print(f"Finished extracting {len(valid_samples)} random samples")
    return valid_samples

In [ ]:
class ContrastiveDecodingLogitsProcessor(LogitsProcessor):
    def __init__(self, amateur_decoder, alpha=0.1, amateur_temperature=1.0, apply_to_codebooks=[0,1,2,3]):
        self.amateur_decoder = amateur_decoder
        self.alpha = alpha
        self.amateur_temperature = amateur_temperature
        self.apply_to_codebooks = apply_to_codebooks
        self.reset()

    def set_amateur_context(self, encoder_hidden_states, attention_mask):
        self.amateur_encoder_hidden_states = encoder_hidden_states
        self.amateur_encoder_attention_mask = attention_mask

    def reset(self):
        self.past_key_values = None

    def __call__(self, input_ids: torch.LongTensor, expert_logits: torch.FloatTensor) -> torch.FloatTensor:
        original_dtype = expert_logits.dtype
        amateur_input_ids = input_ids.to(self.amateur_decoder.device)
        current_seq_len = amateur_input_ids.shape[1]
        actual_bsz_times_4 = amateur_input_ids.shape[0]

        amateur_attention_mask = torch.ones(
            (actual_bsz_times_4, current_seq_len),
            device=amateur_input_ids.device,
            dtype=torch.long
        )

        with torch.no_grad():
            if self.past_key_values is None:
                amateur_outputs = self.amateur_decoder(
                    input_ids=amateur_input_ids,
                    attention_mask=amateur_attention_mask,
                    encoder_hidden_states=self.amateur_encoder_hidden_states,
                    encoder_attention_mask=self.amateur_encoder_attention_mask,
                    use_cache=True
                )
            else:
                amateur_outputs = self.amateur_decoder(
                    input_ids=amateur_input_ids[:, -1:],
                    attention_mask=amateur_attention_mask,
                    encoder_hidden_states=self.amateur_encoder_hidden_states,
                    encoder_attention_mask=self.amateur_encoder_attention_mask,
                    past_key_values=self.past_key_values,
                    use_cache=True
                )

            self.past_key_values = amateur_outputs.past_key_values

            amateur_logits = amateur_outputs.logits[:, -1, :].to(expert_logits.device, dtype=torch.float32)
            amateur_logits = amateur_logits / self.amateur_temperature

        expert_logits_fp32 = expert_logits.to(torch.float32)
        expert_probs = F.softmax(expert_logits_fp32, dim=-1)
        amateur_probs = F.softmax(amateur_logits, dim=-1)

        max_expert_probs, _ = torch.max(expert_probs, dim=-1, keepdim=True)
        plausibility_mask = expert_probs >= (self.alpha * max_expert_probs)

        eps = 1e-10
        cd_scores = torch.log(expert_probs + eps) - torch.log(amateur_probs + eps)
        cd_scores = cd_scores.masked_fill(~plausibility_mask, float('-inf'))

        invalid_rows = torch.isnan(cd_scores).any(dim=-1) | (cd_scores == float('-inf')).all(dim=-1)
        cd_scores[invalid_rows] = expert_logits_fp32[invalid_rows]

        hybrid_logits = expert_logits_fp32.clone()

        cb_mask = torch.zeros(actual_bsz_times_4, dtype=torch.bool, device=hybrid_logits.device)
        indices = torch.arange(actual_bsz_times_4, device=hybrid_logits.device)

        for cb in self.apply_to_codebooks:
            cb_mask |= (indices % 4 == cb)

        hybrid_logits[cb_mask] = cd_scores[cb_mask]

        return hybrid_logits.to(original_dtype)

In [ ]:
class ContrastiveExperimentPipeline:
    def __init__(self, expert_id, amateur_id):
        self.expert_id = expert_id
        self.amateur_id = amateur_id
        self.family = "musicgen"
        self.temp = 1.0

        self.processor = AutoProcessor.from_pretrained(expert_id)

        self.expert_model = MusicgenForConditionalGeneration.from_pretrained(
            expert_id, device_map="auto", torch_dtype=torch.float16,
        )
        self.amateur_model = MusicgenForConditionalGeneration.from_pretrained(
            amateur_id, device_map="auto", torch_dtype=torch.float16,
        )

        self.cd_processor = ContrastiveDecodingLogitsProcessor(
            amateur_decoder=self.amateur_model.decoder,
            alpha=0.1,
            amateur_temperature=self.temp,
            apply_to_codebooks=[0]
        )

        self.logits_processor = LogitsProcessorList([self.cd_processor])

    def generate_all_baselines(self, prompt_tokens_list, text_prompts):
        lengths = [len(p[0]) for p in prompt_tokens_list]
        assert len(set(lengths)) == 1, f"Prompt lengths not uniform: {lengths}"

        batch_size = len(prompt_tokens_list)
        num_codebooks = 4

        text_inputs = self.processor(text=text_prompts, padding=True, return_tensors="pt")
        text_inputs = {k: v.to(self.expert_model.device) for k, v in text_inputs.items()}

        prompt_tokens = torch.tensor(prompt_tokens_list, device=self.expert_model.device)
        pad_id = self.expert_model.generation_config.pad_token_id
        prompt_tokens = prompt_tokens.masked_fill(prompt_tokens >= pad_id, 0)
        decoder_input_ids = prompt_tokens.view(batch_size * num_codebooks, -1)

        with torch.no_grad():
            exp_enc_out = self.expert_model.text_encoder(**text_inputs)
            exp_hidden = exp_enc_out.last_hidden_state

            ama_inputs = self.processor(text=text_prompts, padding=True, return_tensors="pt")
            ama_inputs = {k: v.to(self.amateur_model.device) for k, v in ama_inputs.items()}
            ama_enc_out = self.amateur_model.text_encoder(**ama_inputs)
            ama_hidden = ama_enc_out.last_hidden_state

            if self.amateur_model.enc_to_dec_proj is not None:
                ama_hidden = self.amateur_model.enc_to_dec_proj(ama_hidden)

        self.cd_processor.set_amateur_context(
            encoder_hidden_states=ama_hidden,
            attention_mask=ama_inputs["attention_mask"]
        )

        decoding_strategies = {
            "Greedy": {"do_sample": False},
            "Top-k": {"do_sample": True, "top_k": 50},
            "Nucleus": {"do_sample": True, "top_p": 0.95},
            "Typical": {"do_sample": True, "typical_p": 0.95},
            "CD": {"do_sample": True, "top_k": 50, "logits_processor": self.logits_processor}
        }

        results = {name: [] for name in decoding_strategies.keys()}

        original_decode = self.expert_model.audio_encoder.decode
        def mock_decode(output_ids_chunk, audio_scales=None):
            class MockOutput:
                audio_values = output_ids_chunk
            return MockOutput()

        self.expert_model.audio_encoder.decode = mock_decode

        for name, kwargs in decoding_strategies.items():
            if "logits_processor" in kwargs:
                self.cd_processor.apply_to_codebooks = [0, 1, 2, 3]
                self.cd_processor.reset()

            outputs = self.expert_model.generate(
                        encoder_outputs=BaseModelOutput(last_hidden_state=exp_hidden),
                        attention_mask=text_inputs["attention_mask"],
                        decoder_input_ids=decoder_input_ids,
                        min_new_tokens=500,
                        max_new_tokens=500,
                        guidance_scale=1.0,
                        **kwargs
            )

            tokens = outputs[0]

            final_continuations = []
            for b in range(batch_size):
                batch_streams = []
                for c in range(num_codebooks):
                    stream = tokens[b, c, -500:].tolist()
                    batch_streams.append(stream)
                final_continuations.append(batch_streams)

            results[name] = final_continuations

        self.expert_model.audio_encoder.decode = original_decode
        return results

    def codebook_experiment(self, prompt_tokens_list, text_prompts):
        lengths = [len(p[0]) for p in prompt_tokens_list]
        assert len(set(lengths)) == 1, f"Prompt lengths not uniform: {lengths}"

        batch_size = len(prompt_tokens_list)
        num_codebooks = 4

        text_inputs = self.processor(text=text_prompts, padding=True, return_tensors="pt")
        text_inputs = {k: v.to(self.expert_model.device) for k, v in text_inputs.items()}

        prompt_tokens = torch.tensor(prompt_tokens_list, device=self.expert_model.device)
        pad_id = self.expert_model.generation_config.pad_token_id
        prompt_tokens = prompt_tokens.masked_fill(prompt_tokens >= pad_id, 0)
        decoder_input_ids = prompt_tokens.view(batch_size * num_codebooks, -1)

        with torch.no_grad():
            exp_enc_out = self.expert_model.text_encoder(**text_inputs)
            exp_hidden = exp_enc_out.last_hidden_state

            ama_inputs = self.processor(text=text_prompts, padding=True, return_tensors="pt")
            ama_inputs = {k: v.to(self.amateur_model.device) for k, v in ama_inputs.items()}
            ama_enc_out = self.amateur_model.text_encoder(**ama_inputs)
            ama_hidden = ama_enc_out.last_hidden_state

            if self.amateur_model.enc_to_dec_proj is not None:
                ama_hidden = self.amateur_model.enc_to_dec_proj(ama_hidden)

        self.cd_processor.set_amateur_context(
            encoder_hidden_states=ama_hidden,
            attention_mask=ama_inputs["attention_mask"]
        )

        cd_strategies = {
            "CD_0": {"do_sample": True, "top_k": 50, "codebooks": [0]},
            "CD_123": {"do_sample": True, "top_k": 50, "codebooks": [1, 2, 3]},
            "CD_all": {"do_sample": True, "top_k": 50, "codebooks": [0, 1, 2, 3]}
            "CD_1": {"do_sample": True, "top_k": 50, "codebooks": [1]},
            "CD_2": {"do_sample": True, "top_k": 50, "codebooks": [2]},
            "CD_3": {"do_sample": True, "top_k": 50, "codebooks": [3]},

        }

        results = {name: [] for name in cd_strategies.keys()}

        original_decode = self.expert_model.audio_encoder.decode
        def mock_decode(output_ids_chunk, audio_scales=None):
            class MockOutput:
                audio_values = output_ids_chunk
            return MockOutput()

        self.expert_model.audio_encoder.decode = mock_decode

        for name, kwargs in cd_strategies.items():
            target_codebooks = kwargs.pop("codebooks")
            self.cd_processor.apply_to_codebooks = target_codebooks
            self.cd_processor.reset()

            outputs = self.expert_model.generate(
                        encoder_outputs=BaseModelOutput(last_hidden_state=exp_hidden),
                        attention_mask=text_inputs["attention_mask"],
                        decoder_input_ids=decoder_input_ids,
                        min_new_tokens=500,
                        max_new_tokens=500,
                        guidance_scale=1.0,
                        logits_processor=self.logits_processor,
                        **kwargs
            )

            tokens = outputs[0]

            final_continuations = []
            for b in range(batch_size):
                batch_streams = []
                for c in range(num_codebooks):
                    stream = tokens[b, c, -500:].tolist()
                    batch_streams.append(stream)
                final_continuations.append(batch_streams)

            results[name] = final_continuations

        self.expert_model.audio_encoder.decode = original_decode
        return results

    def prompt_experiment(self, prompt_tokens_list, text_prompts, anti_text_prompts):
        lengths = [len(p[0]) for p in prompt_tokens_list]
        assert len(set(lengths)) == 1, f"Prompt lengths not uniform: {lengths}"

        batch_size = len(prompt_tokens_list)
        num_codebooks = 4

        text_inputs = self.processor(text=text_prompts, padding=True, return_tensors="pt")
        text_inputs = {k: v.to(self.expert_model.device) for k, v in text_inputs.items()}

        prompt_tokens = torch.tensor(prompt_tokens_list, device=self.expert_model.device)
        pad_id = self.expert_model.generation_config.pad_token_id
        prompt_tokens = prompt_tokens.masked_fill(prompt_tokens >= pad_id, 0)
        decoder_input_ids = prompt_tokens.view(batch_size * num_codebooks, -1)

        with torch.no_grad():
            exp_enc_out = self.expert_model.text_encoder(**text_inputs)
            exp_hidden = exp_enc_out.last_hidden_state

        cd_strategies = {
            "CD_anti_genre_prompt": {"do_sample": True, "top_k": 50, "prompt_type": "anti"},
            "CD_no_prompt": {"do_sample": True, "top_k": 50, "prompt_type": "none"}
        }

        results = {name: [] for name in cd_strategies.keys()}

        original_decode = self.expert_model.audio_encoder.decode
        def mock_decode(output_ids_chunk, audio_scales=None):
            class MockOutput:
                audio_values = output_ids_chunk
            return MockOutput()

        self.expert_model.audio_encoder.decode = mock_decode

        for name, kwargs in cd_strategies.items():
            prompt_type = kwargs.pop("prompt_type")

            if prompt_type == "anti":
                ama_texts = anti_text_prompts
            elif prompt_type == "none":
                ama_texts = [""] * batch_size

            with torch.no_grad():
                ama_inputs = self.processor(text=ama_texts, padding=True, return_tensors="pt")
                ama_inputs = {k: v.to(self.amateur_model.device) for k, v in ama_inputs.items()}
                ama_enc_out = self.amateur_model.text_encoder(**ama_inputs)
                ama_hidden = ama_enc_out.last_hidden_state

                if self.amateur_model.enc_to_dec_proj is not None:
                    ama_hidden = self.amateur_model.enc_to_dec_proj(ama_hidden)

            self.cd_processor.set_amateur_context(
                encoder_hidden_states=ama_hidden,
                attention_mask=ama_inputs["attention_mask"]
            )

            self.cd_processor.apply_to_codebooks = [0, 1, 2, 3] # [1, 2, 3]
            self.cd_processor.reset()

            outputs = self.expert_model.generate(
                        encoder_outputs=BaseModelOutput(last_hidden_state=exp_hidden),
                        attention_mask=text_inputs["attention_mask"],
                        decoder_input_ids=decoder_input_ids,
                        min_new_tokens=500,
                        max_new_tokens=500,
                        guidance_scale=1.0,
                        logits_processor=self.logits_processor,
                        **kwargs
            )

            tokens = outputs[0]

            final_continuations = []
            for b in range(batch_size):
                batch_streams = []
                for c in range(num_codebooks):
                    stream = tokens[b, c, -500:].tolist()
                    batch_streams.append(stream)
                final_continuations.append(batch_streams)

            results[name] = final_continuations

        self.expert_model.audio_encoder.decode = original_decode
        return results


    def cleanup(self):
        import gc
        del self.expert_model
        del self.amateur_model
        gc.collect()
        torch.cuda.empty_cache()

In [ ]:
def run_experiment(pipeline, data, experiment="baseline", batch_size=8):
    safe_expert = pipeline.expert_id.split('/')[-1]
    safe_amateur = pipeline.amateur_id.split('/')[-1]

    base_path = f"/content/drive/{experiment}/"
    os.makedirs(base_path, exist_ok=True)

    file_path = os.path.join(base_path, f"{safe_expert}_vs_{safe_amateur}.jsonl")

    print(f"Saving to: {file_path}")

    with open(file_path, "a", encoding="utf-8") as f:
        for i in range(0, len(data), batch_size):
            batch = data[i : i + batch_size]
            text_prompts = [pair["text_prompt"] for pair in batch]
            prompts = [pair["prompt"] for pair in batch]
            references = [pair["reference"] for pair in batch]

            if experiment == "baseline":
                generations = pipeline.generate_all_baselines(prompts, text_prompts)

            elif experiment == "codebook":
                generations = pipeline.codebook_experiment(prompts, text_prompts)

            elif experiment == "prompt":
                anti_prompts = [pair["anti_text_prompt"] for pair in batch]
                generations = pipeline.prompt_experiment(prompts, text_prompts, anti_prompts)

            for j in range(len(batch)):
                record = {
                    "id": i + j,
                    "prompt": prompts[j],
                    "human_reference": references[j],
                    "generations": {method: text_list[j] for method, text_list in generations.items()}
                }
                f.write(json.dumps(record) + "\n")

            f.flush()
            os.fsync(f.fileno())
            print(f"Saved {min(i + batch_size, len(data))}/{len(data)} samples")

    print("Finished\n")

In [ ]:
pipeline = ContrastiveExperimentPipeline(
    expert_id="facebook/musicgen-large",
    amateur_id="facebook/musicgen-small",
)

TARGET_SAMPLES = 1000
gtzan_samples = get_ds(
    dataset_path="sanchit-gandhi/gtzan",
    dataset_name=None,
    split="train",
    target_count=TARGET_SAMPLES,
    processor=pipeline.processor,
    expert_model=pipeline.expert_model
)

run_experiment(pipeline, gtzan_samples, experiment="baseline", batch_size=64)
run_experiment(pipeline, gtzan_samples, experiment="codebook", batch_size=64)
run_experiment(pipeline, gtzan_samples, experiment="prompt", batch_size=64)




preprocessor_config.json:   0%|          | 0.00/275 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/2.36G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/611 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/224 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/703 [00:00<?, ?B/s]

Finished extracting 999 random samples
Saving to: /content/drive/MyDrive/CS4782-final/music-gen/anti-prompt-on-123-prompt/musicgen-large_vs_musicgen-small.jsonl
Saved 64/999 samples
Saved 128/999 samples
Saved 192/999 samples
Saved 256/999 samples
Saved 320/999 samples
Saved 384/999 samples
Saved 448/999 samples
Saved 512/999 samples
Saved 576/999 samples
Saved 640/999 samples
Saved 704/999 samples
Saved 768/999 samples
Saved 832/999 samples
Saved 896/999 samples
Saved 960/999 samples
Saved 999/999 samples
Finished

